Imports de bibliotecas

In [1]:
# pandas for data manipulation
import pandas as pd

# numpy for numerical operations
import numpy as np

# pyplot for plotting
import matplotlib.pyplot as plt

# re for regular expressions
import re

# sklearn for machine learning
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import sklearn as skl

# joblib for saving/loading models
import joblib

# other .py files
from csv_import import get_csv_files

Retrieve CSV files

In [2]:
path = "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\CSI DATA D&M"

files_csi, files_diana, files_afinar = get_csv_files(path)

In [3]:
def print_useful_info(file):
	# Print information about the files dictionaries
	print("Número de ficheiros recolhidos em ", file, ":", len(file))
	print("ESPs por cenário:", len(next(iter(file.values()))))

	# Print all scenarios names
	print("\nTodos os cenários:", sorted(file.keys()))

	# Count scenarios by letter prefix
	scenario_prefixes = {}
	for scenario in file:
		prefix = scenario[0]
		scenario_prefixes[prefix] = scenario_prefixes.get(prefix, 0) + 1

	print("\nScenarios count by prefix:")
	for prefix, count in sorted(scenario_prefixes.items()):
		print(f"Prefix {prefix}: {count} scenarios")

	print("\n")

In [4]:
print_useful_info(files_csi)
print_useful_info(files_diana)
print_useful_info(files_afinar)

Número de ficheiros recolhidos em  {'sem_pessoa': {'esp1': 'C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\CSI DATA D&M\\user00_positionz00_esp01_2025-04-01_02.csv', 'esp2': 'C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\CSI DATA D&M\\user00_positionz00_esp02_2025-04-01_02.csv', 'esp3': 'C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\CSI DATA D&M\\user00_positionz00_esp03_2025-04-01_02.csv', 'esp4': 'C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\CSI DATA D&M\\user00_positionz00_esp04_2025-04-01_02.csv'}, 'sem_pessoa_1': {'esp1': 'C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\CSI DATA D&M\\00_z00_esp1_2025-06-24.csv', 'esp2': 'C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project\\CSI DATA D

In [5]:
# função que processo o arquivo CSV completo
# # e extrai a info da coluna do CSI
# # converte em números complexos
# # calcula a magnitude (2.º passo)
# # 
# e faz ainda uma 1a limpeza
# # remove 2 primeiras subportadoras
# # remove colunas a zero
# # aplica FFT shift

# da tese
# # Vector size verification
# # Subcarrier layout and magnitude computation

def process_csi(file):
    df = pd.read_csv(file, header=None)
    csi_col = df.iloc[:, 26]

    valid_csi = []
    for entry in csi_col:
        match = re.search(r'\[(.*?)\]', str(entry))
        if not match:
            continue
        nums = [float(n) for n in re.findall(r'-?\d+', match.group(1))]
        if len(nums) == 128:
            valid_csi.append(nums)

    valid_csi = np.array(valid_csi)
    complex_csi = valid_csi[:, ::2] + 1j * valid_csi[:, 1::2]
    magnitudes = np.abs(complex_csi)

    # Limpeza: remover 2 primeiras subportadoras e colunas todas-zero
    magnitudes = magnitudes[:, 2:]
    magnitudes = magnitudes[:, ~np.all(magnitudes == 0, axis=0)]
    magnitudes = np.fft.fftshift(magnitudes, axes=1)

    return magnitudes

# cria o dict magnitudes
# itera sobre cada posicao da grelha
# # para cada posicao, itera sobre cada esp_id e path
# # devolve o dict magnitudes preenchido
def process_magnitudes(file):
    
    magnitudes = {}

    for posicao_grelha in file:
        magnitudes[posicao_grelha] = {}
        
        # dict com "esp_id" e "path"
        esps_do_posicao = file[posicao_grelha]
        
        for esp_id, ficheiro in esps_do_posicao.items():
            dados_processados = process_csi(ficheiro)
            magnitudes[posicao_grelha][esp_id] = dados_processados

    return magnitudes

# o dict magnitudes fica
# posicao1
# # esp1
# # # array de arrays (cada leitura é um array)
# # ...
# # esp4
# # # array de arrays (cada leitura é um array)

In [6]:
magnitudes = process_magnitudes(files_csi)
magnitudes_diana = process_magnitudes(files_diana)
magnitudes_afinadas = process_magnitudes(files_afinar)

In [7]:
# cálculo das médias das subportadoras

# função que extrai valores de determiada subportadora
# # recolhe apenas uma fracção dos valores (1/6 * nº leituras)
# # e devolve um dicionário com os resultados
# # # dict esp: array de valores
def extrai_valores_sc(magnitudes_dict, posicao, subcarrier_index, frac):
    resultados = {}
    for esp, matriz in magnitudes_dict[posicao].items():
        n = int(matriz.shape[0] * frac)
        resultados[esp] = matriz[:n, subcarrier_index]
    return resultados

# função que calcula a média dos valores extraídos na função de cima
# retorna um dict com 
# # esp: média
def calcular_media_sc(subportadora_dict):
    return {esp: np.mean(valores) for esp, valores in subportadora_dict.items()}

# função que calcula a média para cada subportadora
# # para cada sc
# # extrai os (n * frac) primeiros valores de cada esp
# # calcula a sua média 
# # # devolve um dict com
# # # subcarrier: {esp_y: média}
def calcular_medias_para_varias_subportadoras(magnitudes_dict, posicao, subcarrier_indices, frac):
    medias_por_subc = {}
    for subc in subcarrier_indices:
        valores = extrai_valores_sc(magnitudes_dict, posicao, subc, frac)
        medias = calcular_media_sc(valores)
        medias_por_subc[subc] = medias
    return medias_por_subc

In [8]:
# Grupo: c06
#   esp: (nº leituras, nº subportadoras)
#   esp1: (76, 51) # 76 é matriz.shape[0]
#   esp2: (71, 51)
#   esp3: (79, 51)
#   esp4: (99, 51)

# posição para retirar normalizações
posicao = 'c06'

# inicializa lista de subportadoras e números de ESPs
subcarriers = list(range(51))
esp_id = [1, 2, 3, 4]
frac = 1/6

medias_c06 = calcular_medias_para_varias_subportadoras(magnitudes, posicao, subcarriers, frac)
# para a posição c06
# # para a subcarrier_x
# # # esp1: média
# # # esp2: média
# # # esp3: média
# # # esp4: média

# Gera o dicionário final no formato usado no Raspberry Pi
normalization_means_full = {
    esp: {
        subc: medias_c06[subc][f"esp{esp}"]
        for subc in range(51)
    }
    for esp in esp_id
}
#joblib.dump(normalization_means_full, 'normalization_means_full.pkl')

Criar dataset "dados" (sc x posição) (array)

Ao invés de "magnitudes" (posição x esp) (array de arrays)


In [9]:
# Parâmetros
N = 90
M = 350
O = 60
A = 292
I = 200

tamanhos = {
    'sem_pessoa': M, 'sem_pessoa_1': M,
    'a00': N, 'c05': N, 'a01': N, 'c06': N,
    'a09': N, 'c07': N, 'a10': N, 'c08': N,
    'a11': N, 'c09': N, 'b00': N, 'c10': N,
    'b01': N, 'c11': N, 'b02': N, 'd01': N,
    'b03': N, 'd02': N, 'b04': N, 'd03': N,
    'b05': N, 'd04': N, 'b06': N, 'd05': N,
    'b07': N, 'd06': N, 'b08': N, 'd07': N,
    'b09': N, 'd08': N, 'b10': N, 'd09': N,
    'b11': N, 'd10': N, 'c01': N, 'd11': N,
    'c02': N, 'e04': N, 'c03': N, 'e05': N,
    'c04': N, 'e06': N,
}

tamanhos_diana = {
    'sem_pessoa': M, 'sem_pessoa_1': M,
    'a00': O, 'c05': O, 'a01': O, 'c06': O,
    'a09': O, 'c07': O, 'a10': O, 'c08': O,
    'a11': O, 'c09': O, 'b00': O, 'c10': O,
    'b01': O, 'c11': O, 'b02': O, 'd01': O,
    'b03': O, 'd02': O, 'b04': O, 'd03': O,
    'b05': O, 'd04': O, 'b06': O, 'd05': O,
    'b07': O, 'd06': O, 'b08': O, 'd07': O,
    'b09': O, 'd08': O, 'b10': O, 'd09': O,
    'b11': O, 'd10': O, 'c01': O, 'd11': O,
    'c02': O, 'e04': O, 'c03': O, 'e05': O,
    'c04': O, 'e06': O,
}

tamanhos_afinar = {
    'com_pessoa': A, 'sem_pessoa': A, 'com_pessoa_2': I, 'sem_pessoa_2': I,
}

posicoes = ['sem_pessoa', 'sem_pessoa_1', 'a00', 'a01', 'a09', 'a10', 'a11', 'b00', 'b01', 'b02',
        'b03', 'b04', 'b05', 'b06', 'b07', 'b08', 'b09', 'b10', 'b11', 'c01', 'c02', 'c03',
        'c04', 'c05', 'c06', 'c07', 'c08', 'c09', 'c10', 'c11', 'd01', 'd02', 'd03',
        'd04', 'd05', 'd06', 'd07', 'd08', 'd09', 'd10', 'd11', 'e04', 'e05', 'e06']

posicoes_afinar = ['sem_pessoa', 'sem_pessoa_2', 'com_pessoa', 'com_pessoa_2']

In [10]:
# para cada posição
# # ver o tamanho de cada (auto definido em cima)
# # e para cada subportadora
# # # e para cada esp
# # # # extrair o vetor correto
# # # # guardar no dicionário
def criar_dataset(posicoes, tamanhos, magnitudes):
    dados = {subc: {} for subc in subcarriers}
    for posicao in posicoes:
        tam = tamanhos[posicao]
        for subc in subcarriers:
            for i in esp_id:
                # acede ao array de arrays
                matriz = magnitudes[posicao][f'esp{i}']
                # extrai o vetor coluna
                # com (linhas/leituras = tam)
                # com (coluna = subc)
                vetor = matriz[:tam, subc]
                chave = f"{posicao}_{i}"
                dados[subc][chave] = vetor

    return dados

In [11]:
dados = criar_dataset(posicoes, tamanhos, magnitudes)
dados_diana = criar_dataset(posicoes, tamanhos_diana, magnitudes_diana)
dados_afinar = criar_dataset(posicoes_afinar, tamanhos_afinar, magnitudes_afinadas)

Normalização de dados

In [12]:
# Função para normalizar
# a função recebe o dicionário de vetores e o dicionário de médias
# e devolve o dicionário de vetores normalizados
# cada vetor é normalizado subtraindo a média e dividindo pela média
def normalizar_vetores(vetores_dict, medias_dict):
    normalizados = {f"subcarrier_{subc}": {} for subc in subcarriers}
    for subc in subcarriers:
        for posicao in posicoes:
            for id in esp_id:
                chave = f"{posicao}_{id}"
                esp = f"esp{id}"
                vetor = vetores_dict[subc][chave]
                media = medias_dict[subc][esp]
                normalizados[f"subcarrier_{subc}"][chave] = (vetor - media) / media
    return normalizados

# Função para normalizar dados a afinar
def normalizar_vetores_afinar(vetores_dict, medias_dict):
    normalizados = {f"subcarrier_{subc}": {} for subc in subcarriers}
    for subc in subcarriers:
        for posicao in posicoes_afinar:
            for id in esp_id:
                chave = f"{posicao}_{id}"
                esp = f"esp{id}"
                vetor = vetores_dict[subc][chave]
                media = medias_dict[subc][esp]
                normalizados[f"subcarrier_{subc}"][chave] = (vetor - media) / media
    return normalizados

In [ ]:
# Aplicar normalização
normalizados_loureiro = normalizar_vetores(dados, medias_c06)
normalizados_diana = normalizar_vetores(dados_diana, medias_c06)
normalizados_afinar = normalizar_vetores_afinar(dados_afinar, medias_c06)


Média, desvio padrão e valor máximo

In [14]:
# função para calcular a média em janelas sobrepostas
def media_grupos_overlap(vetor, tamanho_janela, step):
    medias = []
    for i in range(0, len(vetor) - tamanho_janela + 1, step):
        janela = vetor[i:i + tamanho_janela]
        medias.append(np.mean(janela))
    return np.array(medias)

# função para calcular o desvio padrão em janelas sobrepostas
def std_grupos_overlap(vetor, tamanho_janela, step):
    desvios = []
    for i in range(0, len(vetor) - tamanho_janela + 1, step):
        janela = vetor[i:i + tamanho_janela]
        desvios.append(np.std(janela))
    return np.array(desvios)

# função para calcular o máximo em janelas sobrepostas,
# evitando repetições dos últimos 3 máximos escolhidos
def max_grupos_overlap(vetor, tamanho_janela, step):
    maximos = []

    for i in range(0, len(vetor) - tamanho_janela + 1, step):
        janela = vetor[i:i + tamanho_janela]
        candidatos = np.sort(np.unique(janela))[::-1]  # do maior para o menor

        # Exclui os 3 últimos valores
        ultimos = set(maximos[-3:])

        escolhido = None
        for val in candidatos:
            if val not in ultimos:
                escolhido = val
                break

        # Se todos os valores estão nos últimos 3, aceita o maior mesmo assim
        if escolhido is None:
            escolhido = candidatos[0]

        maximos.append(escolhido)

    return np.array(maximos)

In [15]:
def media_fixed(vetor, tamanho_janela, W):
    step = (len(vetor) - tamanho_janela) // (W - 1)
    return media_grupos_overlap(vetor, tamanho_janela, step)[:W]

def std_fixed(vetor, tamanho_janela, W):
    step = (len(vetor) - tamanho_janela) // (W - 1)
    return std_grupos_overlap(vetor, tamanho_janela, step)[:W]

def max_fixed(vetor, tamanho_janela, W):
    step = (len(vetor) - tamanho_janela) // (W - 1)
    return max_grupos_overlap(vetor, tamanho_janela, step)[:W]


def media_fixed_af(vetor, tamanho_janela, G):
    step = (len(vetor) - tamanho_janela) // (G - 1)
    return media_grupos_overlap(vetor, tamanho_janela, step)[:G]

def std_fixed_af(vetor, tamanho_janela, G):
    step = (len(vetor) - tamanho_janela) // (G - 1)
    return std_grupos_overlap(vetor, tamanho_janela, step)[:G]

def max_fixed_af(vetor, tamanho_janela, G):
    step = (len(vetor) - tamanho_janela) // (G - 1)
    return max_grupos_overlap(vetor, tamanho_janela, step)[:G]

In [16]:
# --- Médias: sem_pessoa mantém step=3, outros usam media_fixed ---
tamanho_janela = 10
step = 3
W = 27   # nº de janelas fixas para "com pessoa"
G = 60   # nº de janelas fixas para "dados_afinar"

media_loureiro = {subc: {} for subc in subcarriers}
media_diana    = {subc: {} for subc in subcarriers}
media_afinar   = {subc: {} for subc in subcarriers}

std_loureiro = {subc: {} for subc in subcarriers}
std_diana    = {subc: {} for subc in subcarriers}
std_afinar   = {subc: {} for subc in subcarriers}

max_loureiro = {subc: {} for subc in subcarriers}
max_diana    = {subc: {} for subc in subcarriers}
max_afinar   = {subc: {} for subc in subcarriers}

for grupo in posicoes:
    for id in esp_id:
        chave = f"{grupo}_{id}"
        for subc in subcarriers:
            v_l = normalizados_loureiro[f"subcarrier_{subc}"][chave]
            v_d = normalizados_diana[f"subcarrier_{subc}"][chave]

            if grupo.startswith("sem_pessoa"):
                # mantém todas as janelas possíveis, sem truncar
                media_l = media_grupos_overlap(v_l, tamanho_janela, step)
                media_d = media_grupos_overlap(v_d, tamanho_janela, step)
                std_l = std_grupos_overlap(v_l, tamanho_janela, step)
                std_d = std_grupos_overlap(v_d, tamanho_janela, step)
                m_l = max_grupos_overlap(v_l, tamanho_janela, step)
                m_d = max_grupos_overlap(v_d, tamanho_janela, step)
            else:
                # força W janelas idênticas
                media_l = media_fixed(v_l, tamanho_janela, W)
                media_d = media_fixed(v_d, tamanho_janela, W)
                std_l = std_fixed(v_l, tamanho_janela, W)
                std_d = std_fixed(v_d, tamanho_janela, W)
                m_l = max_fixed(v_l, tamanho_janela, W)
                m_d = max_fixed(v_d, tamanho_janela, W)

            media_loureiro[subc][chave] = media_l
            media_diana   [subc][chave] = media_d

            std_loureiro[subc][chave] = std_l
            std_diana   [subc][chave] = std_d

            max_loureiro[subc][chave] = m_l
            max_diana   [subc][chave] = m_d

for grupo in posicoes_afinar:
    for id in esp_id:
        chave = f"{grupo}_{id}"
        for subc in subcarriers:
            v = normalizados_afinar [f"subcarrier_{subc}"][chave]
            media = media_fixed_af(v, tamanho_janela, G)
            std = std_fixed_af(v, tamanho_janela, G)
            max = max_fixed_af(v, tamanho_janela, G)

            media_afinar[subc][chave] = media
            std_afinar[subc][chave] = std
            max_afinar[subc][chave] = max

In [22]:
# --- Parâmetros ---
subcarriers_map = {i+1: sub for i, sub in enumerate(subcarriers)}

# dicionários para guardar cada coluna
col_media, col_std, col_max, col_all = {}, {}, {}, {}

# dicionários para guardar cada coluna
col_media_afinar, col_std_afinar, col_max_afinar, col_all_afinar = {}, {}, {}, {}

for subc_label, subc in subcarriers_map.items():
    for esp in esp_id:
        # nome da coluna
        name_mean = f"Mean{subc_label}_esp{esp}"
        name_std  = f"std{subc_label}_esp{esp}"
        name_max  = f"magmax{subc_label}_esp{esp}"
        name_all = f"all{subc_label}_esp{esp}"

        seq_mean, seq_std, seq_max = [], [], []

        # 1) SEM PESSOA: primeiro Loureiro, depois Diana
        seq_mean.append(media_loureiro[subc][f"sem_pessoa_{esp}"])
        seq_mean.append(media_loureiro[subc][f"sem_pessoa_1_{esp}"])

        seq_mean.append(media_diana[subc][f"sem_pessoa_{esp}"])
        seq_mean.append(media_diana[subc][f"sem_pessoa_1_{esp}"])

        seq_std.append(std_loureiro[subc][f"sem_pessoa_{esp}"])
        seq_std.append(std_loureiro[subc][f"sem_pessoa_1_{esp}"])

        seq_std.append(std_diana[subc][f"sem_pessoa_{esp}"])
        seq_std.append(std_diana[subc][f"sem_pessoa_1_{esp}"])

        seq_max.append(max_loureiro[subc][f"sem_pessoa_{esp}"])
        seq_max.append(max_loureiro[subc][f"sem_pessoa_1_{esp}"])

        seq_max.append(max_diana[subc][f"sem_pessoa_{esp}"])
        seq_max.append(max_diana[subc][f"sem_pessoa_1_{esp}"])


        # 2) COM PESSOA: percorre todos os outros grupos
        for grupo in posicoes:
            if grupo == "sem_pessoa":
                continue
            key = f"{grupo}_{esp}"
            seq_mean.append(media_loureiro[subc][key])
            seq_mean.append(media_diana[subc][key])
            seq_std.append( std_loureiro[subc][key])
            seq_std.append( std_diana[subc][key])
            seq_max.append( max_loureiro[subc][key])
            seq_max.append( max_diana[subc][key])

        # 3) concatena e guarda
        col_media[name_mean] = np.concatenate(seq_mean)
        col_std [name_std ] = np.concatenate(seq_std)
        col_max [name_max ] = np.concatenate(seq_max)

for subc_label, subc in subcarriers_map.items():
    for esp in esp_id:
        # nome da coluna
        name_mean_af = f"Mean{subc_label}_esp{esp}"
        name_std_af  = f"std{subc_label}_esp{esp}"
        name_max_af  = f"magmax{subc_label}_esp{esp}"

        seq_mean, seq_std, seq_max = [], [], []

        for grupo in posicoes_afinar:
            key = f"{grupo}_{esp}"

            seq_mean.append(media_afinar[subc][key])
            seq_std.append(std_afinar[subc][key])
            seq_max.append(max_afinar[subc][key])

        col_media_afinar[name_mean_af] = np.concatenate(seq_mean)
        col_std_afinar[name_std_af] = np.concatenate(seq_std)
        col_max_afinar[name_max_af] = np.concatenate(seq_max)

